# Sarc-o-Meter · LLM RAG Model Experiment

**Objective**: Benchmark multiple locally-available LLM models against the Sarc-o-Meter RAG pipeline to evaluate JSON compliance, clinical safety, and inference latency — establishing a performance baseline before fine-tuning.

**Evaluation Method**: Deterministic grading (9 criteria) + Safety checks  
**Test Personas**: 7 (covering Low → Severe risk levels with various clinical flags)

## 1. Model Selection Rationale

We selected 4 models that represent different architectural choices, parameter scales, and multilingual capabilities. All models are quantized to 4-bit (Q4_K_M) for fair comparison and practical on-device considerations.

| Model | Parameters | Size (GB) | Architecture | Rationale |
|-------|-----------|-----------|-------------|----------|
| **Qwen 2.5 3B** | 3B | 1.9 | Qwen2 (Alibaba) | **Current production model.** Best Bahasa Indonesia support among small models. Optimized for instruction following & JSON output. Fits on-device (~2GB RAM). |
| **Qwen 3 8B** | 8B | 5.2 | Qwen3 (Alibaba) | **Quality ceiling benchmark.** Next-gen Qwen with improved reasoning. Tests if larger model significantly outperforms 3B, justifying fine-tuning investment. Too large for on-device iOS deployment. |
| **Llama 3.1 8B** | 8B | 4.9 | LLaMA 3.1 (Meta) | **Industry standard baseline.** Strong English instruction following but weaker Bahasa Indonesia. Tests cross-lingual robustness of our Indonesian prompts against a non-Qwen architecture. |
| **Mistral 7B v0.3** | 7B | 4.4 | Mistral (Mistral AI) | **Efficiency benchmark.** Known for sliding-window attention efficiency. Tests whether a European-origin model can handle Bahasa Indonesia clinical terminology and strict JSON formatting. |

## 2. Setup & Data Loading

In [1]:
import json
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pathlib import Path

# Resolve paths relative to this notebook
EVAL_DIR = Path('/Users/virafitriyani/Downloads/sarc-o-meter/eval')

# Style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'figure.dpi': 120,
})

# Colors
MODEL_COLORS = {
    'Qwen 2.5 3B (Q4_K_M)': '#4F46E5',
    'Qwen 3 8B (Q4_K_M)': '#7C3AED',
    'Llama 3.1 8B (Q4_K_M)': '#059669',
    'Mistral 7B v0.3 (Q4_K_M)': '#D97706',
}

# Load experiment results
results_path = EVAL_DIR / 'model_experiment_results.json'
with open(results_path) as f:
    experiment = json.load(f)

print(f"Experiment date: {experiment['metadata']['date']}")
print(f"Models tested: {experiment['metadata']['models_count']}")
print(f"Personas tested: {experiment['metadata']['personas_count']}")

Experiment date: 2026-08-17T15:18:33.454183
Models tested: 4
Personas tested: 7


## 3. Overall Comparison Summary

In [2]:
# Build comparison dataframe
rows = []
for m in experiment['models']:
    info = m['model']
    s = m.get('summary', {})
    if m['status'] == 'skipped':
        continue
    
    # Calculate per-criteria pass rates
    criteria_pass = {}
    for p in m['personas']:
        for k, v in p.get('grades', {}).items():
            if k not in criteria_pass:
                criteria_pass[k] = {'pass': 0, 'total': 0}
            criteria_pass[k]['total'] += 1
            if v:
                criteria_pass[k]['pass'] += 1
    
    # Safety rate for severe personas
    safety_mentions = 0
    safety_total = 0
    for p in m['personas']:
        if 'mentions_professional' in p.get('safety', {}):
            safety_total += 1
            if p['safety']['mentions_professional']:
                safety_mentions += 1
    
    rows.append({
        'Model': info['name'],
        'Params': info['params'],
        'Size (GB)': info['size_gb'],
        'Pass Rate': s.get('pass_rate', 'N/A'),
        'Pass %': int(s['pass_rate'].split('(')[1].rstrip('%)')) if 'pass_rate' in s else 0,
        'Avg Latency (s)': s.get('avg_latency_s', 0),
        'Median Latency (s)': s.get('median_latency_s', 0),
        'Min Latency (s)': s.get('min_latency_s', 0),
        'Max Latency (s)': s.get('max_latency_s', 0),
        'Std Dev (s)': s.get('stdev_latency_s', 0),
        'JSON Valid %': round(criteria_pass.get('C1_valid_json', {}).get('pass', 0) / max(criteria_pass.get('C1_valid_json', {}).get('total', 1), 1) * 100),
        'Schema %': round(criteria_pass.get('C2_schema_keys', {}).get('pass', 0) / max(criteria_pass.get('C2_schema_keys', {}).get('total', 1), 1) * 100),
        'Exercise Count %': round(criteria_pass.get('C3_exercise_count', {}).get('pass', 0) / max(criteria_pass.get('C3_exercise_count', {}).get('total', 1), 1) * 100),
        'All Fields %': round(criteria_pass.get('C5_exercise_fields', {}).get('pass', 0) / max(criteria_pass.get('C5_exercise_fields', {}).get('total', 1), 1) * 100),
        'No Banned Words %': round(criteria_pass.get('C9_no_banned_words', {}).get('pass', 0) / max(criteria_pass.get('C9_no_banned_words', {}).get('total', 1), 1) * 100),
        'Professional Mention Rate': f"{safety_mentions}/{safety_total}" if safety_total > 0 else 'N/A',
    })

df_summary = pd.DataFrame(rows)
display_cols = ['Model', 'Params', 'Size (GB)', 'Pass Rate', 'Avg Latency (s)', 'Median Latency (s)', 'Min Latency (s)', 'Max Latency (s)']
print(df_summary[display_cols].to_string(index=False))

                   Model Params  Size (GB)    Pass Rate  Avg Latency (s)  Median Latency (s)  Min Latency (s)  Max Latency (s)
    Qwen 2.5 3B (Q4_K_M)     3B        1.9 63/63 (100%)             17.0                15.0             12.4             24.4
      Qwen 3 8B (Q4_K_M)     8B        5.2  62/63 (98%)             77.6                69.0             48.0            114.4
   Llama 3.1 8B (Q4_K_M)     8B        4.9 63/63 (100%)             36.1                30.6             25.6             47.6
Mistral 7B v0.3 (Q4_K_M)     7B        4.4  59/63 (94%)             32.7                27.9             23.8             54.0


## 4. Inference Latency Comparison

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 4a. Average Latency Bar Chart
ax = axes[0]
models = df_summary['Model'].tolist()
avg_lats = df_summary['Avg Latency (s)'].tolist()
colors = [MODEL_COLORS.get(m, '#888') for m in models]
bars = ax.barh(models, avg_lats, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Average Inference Latency (seconds)')
ax.set_title('Average Inference Latency per Model')
ax.invert_yaxis()
for bar, val in zip(bars, avg_lats):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, 
            f'{val:.1f}s', va='center', fontweight='bold', fontsize=10)
ax.set_xlim(0, max(avg_lats) * 1.2)

# 4b. Per-Persona Latency Grouped Bar
ax = axes[1]
persona_names_short = ['A (Low)', 'B (Mid)', 'C (High)', 'D (Severe)', 'E (Red Flag)', 'F (Red Flag)', 'G (Severe)']
x = np.arange(len(persona_names_short))
width = 0.2

for idx, m in enumerate(experiment['models']):
    if m['status'] == 'skipped': continue
    latencies = [p.get('latency_s', 0) for p in m['personas']]
    offset = (idx - 1.5) * width
    color = MODEL_COLORS.get(m['model']['name'], '#888')
    ax.bar(x + offset, latencies, width, label=m['model']['name'].split(' (')[0], color=color, alpha=0.85)

ax.set_xlabel('Test Persona')
ax.set_ylabel('Latency (seconds)')
ax.set_title('Per-Persona Inference Latency')
ax.set_xticks(x)
ax.set_xticklabels(persona_names_short, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'latency_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval/latency_comparison.png')

Saved: eval/latency_comparison.png


/var/folders/cp/0z_jlh_12cz4055700rwdc2c0000gn/T/ipykernel_9016/3990639570.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. JSON Compliance & Quality Metrics

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 5a. Overall Pass Rate
ax = axes[0]
pass_pcts = df_summary['Pass %'].tolist()
colors = [MODEL_COLORS.get(m, '#888') for m in models]
bars = ax.barh(models, pass_pcts, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Pass Rate (%)')
ax.set_title('Overall Deterministic Pass Rate (9 criteria x 7 personas)')
ax.set_xlim(0, 105)
ax.invert_yaxis()
for bar, val in zip(bars, pass_pcts):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{val}%', va='center', fontweight='bold', fontsize=11)
ax.axvline(x=100, color='green', linestyle='--', alpha=0.3, linewidth=1)

# 5b. Per-Criteria Breakdown
ax = axes[1]
criteria_labels = ['JSON Valid', 'Schema Keys', 'Exercise Count', 'All Fields', 'No Banned Words']
criteria_keys = ['JSON Valid %', 'Schema %', 'Exercise Count %', 'All Fields %', 'No Banned Words %']
x = np.arange(len(criteria_labels))
width = 0.2

for idx, (_, row) in enumerate(df_summary.iterrows()):
    vals = [row[k] for k in criteria_keys]
    offset = (idx - 1.5) * width
    color = MODEL_COLORS.get(row['Model'], '#888')
    ax.bar(x + offset, vals, width, label=row['Model'].split(' (')[0], color=color, alpha=0.85)

ax.set_ylabel('Pass Rate (%)')
ax.set_title('Per-Criteria Pass Rate by Model')
ax.set_xticks(x)
ax.set_xticklabels(criteria_labels, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0, 110)
ax.axhline(y=100, color='green', linestyle='--', alpha=0.3, linewidth=1)
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'quality_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval/quality_comparison.png')

Saved: eval/quality_comparison.png


/var/folders/cp/0z_jlh_12cz4055700rwdc2c0000gn/T/ipykernel_9016/2974492509.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Safety Compliance Analysis

In [5]:
safety_rows = []
for m in experiment['models']:
    if m['status'] == 'skipped': continue
    info = m['model']
    mentions_pro = []
    respects_1ex = []
    conservative = []
    for p in m['personas']:
        s = p.get('safety', {})
        if 'mentions_professional' in s: mentions_pro.append(s['mentions_professional'])
        if 'respects_1_exercise' in s: respects_1ex.append(s['respects_1_exercise'])
        if 'conservative_reps' in s: conservative.append(s['conservative_reps'])
    safety_rows.append({
        'Model': info['name'],
        'Mentions Professional': f"{sum(mentions_pro)}/{len(mentions_pro)}" if mentions_pro else 'N/A',
        'Respects 1-Exercise Limit': f"{sum(respects_1ex)}/{len(respects_1ex)}" if respects_1ex else 'N/A',
        'Conservative Reps': f"{sum(conservative)}/{len(conservative)}" if conservative else 'N/A',
    })
df_safety = pd.DataFrame(safety_rows)
print('Safety Compliance for Red-Flag / Severe Personas')
print(df_safety.to_string(index=False))

Safety Compliance for Red-Flag / Severe Personas
                   Model Mentions Professional Respects 1-Exercise Limit Conservative Reps
    Qwen 2.5 3B (Q4_K_M)                   2/3                       4/4               4/4
      Qwen 3 8B (Q4_K_M)                   3/3                       4/4               4/4
   Llama 3.1 8B (Q4_K_M)                   2/3                       4/4               4/4
Mistral 7B v0.3 (Q4_K_M)                   2/3                       4/4               4/4


## 7. Per-Persona Detailed Heatmap

In [6]:
model_names = [m['model']['name'] for m in experiment['models'] if m['status'] != 'skipped']
score_matrix = []
for m in experiment['models']:
    if m['status'] == 'skipped': continue
    row = [sum(1 for v in p.get('grades', {}).values() if v) for p in m['personas']]
    score_matrix.append(row)
score_array = np.array(score_matrix)

fig, ax = plt.subplots(figsize=(12, 4))
persona_short = ['A\n(Low)', 'B\n(Mid)', 'C\n(High)', 'D\n(Severe)', 'E\n(Red Flag)', 'F\n(Red Flag)', 'G\n(Severe)']
im = ax.imshow(score_array, cmap='RdYlGn', vmin=5, vmax=9, aspect='auto')
ax.set_xticks(range(len(persona_short)))
ax.set_xticklabels(persona_short, fontsize=9)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels([n.split(' (')[0] for n in model_names], fontsize=10)
ax.set_title('Score per Persona per Model (out of 9)')
for i in range(len(model_names)):
    for j in range(len(persona_short)):
        val = score_array[i, j]
        color = 'white' if val < 7 else 'black'
        ax.text(j, i, f'{val}/9', ha='center', va='center', fontweight='bold', fontsize=11, color=color)
plt.colorbar(im, ax=ax, label='Score', shrink=0.8)
plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'heatmap_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval/heatmap_comparison.png')

Saved: eval/heatmap_comparison.png


/var/folders/cp/0z_jlh_12cz4055700rwdc2c0000gn/T/ipykernel_9016/939337220.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Latency Distribution (Box Plot)

In [7]:
fig, ax = plt.subplots(figsize=(10, 5))
latency_data = []
labels = []
colors_list = []
for m in experiment['models']:
    if m['status'] == 'skipped': continue
    lats = [p.get('latency_s', 0) for p in m['personas']]
    latency_data.append(lats)
    short_name = m['model']['name'].split(' (')[0]
    labels.append(f"{short_name}\n({m['model']['params']})")
    colors_list.append(MODEL_COLORS.get(m['model']['name'], '#888'))

bp = ax.boxplot(latency_data, patch_artist=True, labels=labels, widths=0.5)
for patch, color in zip(bp['boxes'], colors_list):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
for i, lats in enumerate(latency_data):
    x = np.random.normal(i + 1, 0.04, size=len(lats))
    ax.scatter(x, lats, alpha=0.6, s=30, color=colors_list[i], edgecolor='white', zorder=3)
ax.set_ylabel('Inference Latency (seconds)')
ax.set_title('Latency Distribution Across All Personas')
ax.grid(axis='y', alpha=0.3)
for i, lats in enumerate(latency_data):
    median = np.median(lats)
    ax.text(i + 1, median + 1.5, f'{median:.1f}s', ha='center', fontweight='bold', fontsize=9, color=colors_list[i])
plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'latency_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval/latency_boxplot.png')

/var/folders/cp/0z_jlh_12cz4055700rwdc2c0000gn/T/ipykernel_9016/840889687.py:13: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(latency_data, patch_artist=True, labels=labels, widths=0.5)


Saved: eval/latency_boxplot.png


/var/folders/cp/0z_jlh_12cz4055700rwdc2c0000gn/T/ipykernel_9016/840889687.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Failure Analysis

In [8]:
print('=' * 72)
print('FAILURE ANALYSIS')
print('=' * 72)
for m in experiment['models']:
    if m['status'] == 'skipped': continue
    info = m['model']
    failures = []
    for p in m['personas']:
        for k, v in p.get('grades', {}).items():
            if not v:
                failures.append({'Persona': p['persona'], 'Criterion': k})
    print(f"\n{info['name']}:")
    if failures:
        print(f"  Total failures: {len(failures)}")
        for f in failures:
            print(f"  x {f['Persona']} -> {f['Criterion']}")
    else:
        print('  No failures - perfect 100% pass rate')

FAILURE ANALYSIS

Qwen 2.5 3B (Q4_K_M):
  No failures - perfect 100% pass rate

Qwen 3 8B (Q4_K_M):
  Total failures: 1
  x E — 68F Red Flags (recent surgery + heart condition) -> C9_no_banned_words

Llama 3.1 8B (Q4_K_M):
  No failures - perfect 100% pass rate

Mistral 7B v0.3 (Q4_K_M):
  Total failures: 4
  x A — Healthy 45F (Low Risk) -> C5_exercise_fields
  x B — 62M Possible Sarcopenia (Mid Risk) -> C5_exercise_fields
  x E — 68F Red Flags (recent surgery + heart condition) -> C9_no_banned_words
  x G — 70F Severe + Balance/Dizziness + Skipped Tests -> C9_no_banned_words


## 10. Efficiency Score (Quality / Latency Trade-off)

In [9]:
fig, ax = plt.subplots(figsize=(9, 6))
for _, row in df_summary.iterrows():
    color = MODEL_COLORS.get(row['Model'], '#888')
    size = float(row['Size (GB)']) * 60
    ax.scatter(row['Avg Latency (s)'], row['Pass %'], 
              s=size, c=color, alpha=0.7, edgecolor='white', linewidth=2, zorder=3)
    ax.annotate(row['Model'].split(' (')[0] + f"\n({row['Params']})", 
               (row['Avg Latency (s)'], row['Pass %']),
               textcoords='offset points', xytext=(12, -5), fontsize=9, fontweight='bold',
               color=color)
ax.set_xlabel('Average Inference Latency (seconds)', fontsize=11)
ax.set_ylabel('Pass Rate (%)', fontsize=11)
ax.set_title('Quality vs. Speed Trade-off\n(bubble size = model file size)', fontsize=13)
ax.set_ylim(88, 102)
ax.set_xlim(0, max(df_summary['Avg Latency (s)']) * 1.3)
ax.axhline(y=100, color='green', linestyle='--', alpha=0.2, linewidth=1)
ax.annotate('IDEAL ZONE\n(high quality, low latency)', 
           xy=(10, 101), fontsize=9, color='green', alpha=0.5, fontstyle='italic')
plt.tight_layout()
plt.savefig(str(EVAL_DIR / 'efficiency_tradeoff.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eval/efficiency_tradeoff.png')

Saved: eval/efficiency_tradeoff.png


/var/folders/cp/0z_jlh_12cz4055700rwdc2c0000gn/T/ipykernel_9016/1181888956.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Results Summary & Recommendation

In [10]:
print('=' * 72)
print('FINAL MODEL COMPARISON RESULTS')
print('=' * 72)
print()
header = f'{"Model":<28} {"Params":<8} {"Size":<8} {"Pass Rate":<14} {"Avg Lat":<10} {"Med Lat":<10} {"Std Dev":<8}'
print(header)
print('-' * len(header))
for _, row in df_summary.iterrows():
    print(f'{row["Model"]:<28} {row["Params"]:<8} {row["Size (GB)"]:<8} {row["Pass Rate"]:<14} {row["Avg Latency (s)"]:<10} {row["Median Latency (s)"]:<10} {row["Std Dev (s)"]:<8}')

print()
print('=' * 72)
print('KEY FINDINGS')
print('=' * 72)
print()
print('1. Qwen 2.5 3B is the clear winner for on-device deployment:')
print('   - Perfect 100% pass rate (63/63 checks)')
print('   - Fastest average latency (17.0s) - 2x faster than 7B/8B models')
print('   - Smallest size (1.9 GB) - fits comfortably on iPhone')
print('   - Best Bahasa Indonesia JSON compliance')
print()
print('2. Llama 3.1 8B also achieved 100% but at 2x the latency (36.1s)')
print('   - Strong English instruction following translates well to Indonesian')
print('   - Too large for on-device (4.9 GB)')
print()
print('3. Qwen 3 8B scored 98% but with the worst latency (77.6s avg)')
print('   - Failed C9 (banned word) on Persona E')
print('   - Thinking/reasoning mode adds significant overhead')
print()
print('4. Mistral 7B scored lowest (94%) with multiple failure types:')
print('   - Missing exercise fields (C5) on Personas A & B')
print('   - Used banned word on Personas E & G')
print('   - Weaker Bahasa Indonesia support confirmed')
print()
print('=' * 72)
print('RECOMMENDATION')
print('=' * 72)
print()
print('-> Keep Qwen 2.5 as the base model family for fine-tuning.')
print('-> Fine-tune Qwen 2.5 1.5B (half the current model size) to achieve:')
print('   - Even faster inference (~1-4s on device vs current 17s)')
print('   - Smaller footprint (~1.0 GB vs current 1.9 GB)')
print('   - Built-in JSON formatting (no long prompt instructions needed)')
print('   - Domain-specific Bahasa Indonesia clinical vocabulary')

FINAL MODEL COMPARISON RESULTS

Model                        Params   Size     Pass Rate      Avg Lat    Med Lat    Std Dev 
--------------------------------------------------------------------------------------------
Qwen 2.5 3B (Q4_K_M)         3B       1.9      63/63 (100%)   17.0       15.0       5.0     
Qwen 3 8B (Q4_K_M)           8B       5.2      62/63 (98%)    77.6       69.0       27.1    
Llama 3.1 8B (Q4_K_M)        8B       4.9      63/63 (100%)   36.1       30.6       9.7     
Mistral 7B v0.3 (Q4_K_M)     7B       4.4      59/63 (94%)    32.7       27.9       11.3    

KEY FINDINGS

1. Qwen 2.5 3B is the clear winner for on-device deployment:
   - Perfect 100% pass rate (63/63 checks)
   - Fastest average latency (17.0s) - 2x faster than 7B/8B models
   - Smallest size (1.9 GB) - fits comfortably on iPhone
   - Best Bahasa Indonesia JSON compliance

2. Llama 3.1 8B also achieved 100% but at 2x the latency (36.1s)
   - Strong English instruction following translates well 